# Pareto Fronts: Runtime vs Accuracy per Drift Detector

Reads `<Detector>_<Dataset>.csv` files produced by `parse_slurm_out.py` and plots the Pareto front (runtime vs accuracy) for each detector, grouped by dataset.

In [1]:
import os
import glob
import re
from collections import defaultdict

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
%matplotlib ipympl

In [3]:
#%pip list | grep ipywidgets

^C
ERROR: Operation cancelled by user
Note: you may need to restart the kernel to use updated packages.


In [4]:
#%pip install ipympl==0.7.0

In [ ]:
# Directory containing the CSV files from parse_slurm_out.py
CSV_DIR = 'results'

# Discover all <Detector>_<Dataset>.csv files
csv_files = glob.glob(os.path.join(CSV_DIR, '*.csv'))

# Parse filenames into (detector, dataset) and load DataFrames
# Expected filename pattern: <Detector>_<Dataset>.csv
# Also handles: <Detector>_<Dataset>_PreSelected.csv -> detector='MoPEDDs_PreSelected'
# EWDD is displayed as MoPEDDs
data = defaultdict(dict)  # data[dataset][detector] = DataFrame

# Display name mapping (filename detector -> plot label)
DISPLAY_NAMES = {
    'EWDD': 'MoPEDDs',
}

for path in sorted(csv_files):
    fname = os.path.basename(path)
    # Try PreSelected pattern first: e.g. EWDD_Electricity_PreSelected.csv
    match_pre = re.match(r'^(.+?)_(.+)_PreSelected\.csv$', fname)
    if match_pre:
        detector = 'MoPEDDs_PreSelected'
        dataset = match_pre.group(2)
    else:
        match = re.match(r'^(.+?)_(.+)\.csv$', fname)
        if not match:
            continue
        detector, dataset = match.group(1), match.group(2)
        detector = DISPLAY_NAMES.get(detector, detector)
    df = pd.read_csv(path)
    if 'accuracy' in df.columns and 'runtime' in df.columns:
        data[dataset][detector] = df

print(f'Found {sum(len(v) for v in data.values())} CSV files across {len(data)} dataset(s):')
for ds in sorted(data):
    detectors = sorted(data[ds])
    print(f'  {ds}: {detectors}')

In [4]:
def compute_pareto_front(df):
    """Compute the Pareto front for maximizing accuracy and minimizing runtime.
    
    A trial is Pareto-optimal if no other trial has both higher accuracy AND lower runtime.
    Returns a DataFrame sorted by accuracy (ascending) for plotting.
    """
    points = df[['accuracy', 'runtime']].dropna().copy()
    points = points.sort_values('accuracy', ascending=False).reset_index(drop=True)
    
    pareto = []
    min_runtime = float('inf')
    
    for _, row in points.iterrows():
        if row['runtime'] <= min_runtime:
            pareto.append(row)
            min_runtime = row['runtime']
    
    pareto_df = pd.DataFrame(pareto).sort_values('accuracy').reset_index(drop=True)
    return pareto_df

In [ ]:
# Color map for detectors
DETECTOR_COLORS = {
    'BNDM': '#1f77b4',
    'CSDDM': '#ff7f0e',
    'D3': '#2ca02c',
    'IBDD': '#d62728',
    'OCDD': '#9467bd',
    'SPLL': '#8c564b',
    'UDetect': '#e377c2',
    'MoPEDDs': '#17becf',
    'MoPEDDs_PreSelected': '#bcbd22',
}

DETECTOR_MARKERS = {
    'BNDM': 'o',
    'CSDDM': 's',
    'D3': '^',
    'IBDD': 'D',
    'OCDD': 'v',
    'SPLL': 'P',
    'UDetect': 'X',
    'MoPEDDs': '*',
    'MoPEDDs_PreSelected': 'h',
}

In [ ]:
for dataset in sorted(data):
    detectors = data[dataset]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for detector_name in sorted(detectors):
        df = detectors[detector_name]
        n_successful = len(df[df['runtime'] != float('inf')])
        n_total = len(df)
        color = DETECTOR_COLORS.get(detector_name, '#333333')
        marker = DETECTOR_MARKERS.get(detector_name, 'o')
        
        # Plot all trials as faint scatter
        ax.scatter(
            df['runtime'], df['accuracy'],
            color=color, marker=marker, alpha=0.15, s=20,
        )
        
        # Compute and plot Pareto front
        pareto = compute_pareto_front(df)
        ax.plot(
            pareto['runtime'], pareto['accuracy'],
            color=color, linewidth=2, label=f'{detector_name} ({n_successful} successful runs ({n_total} trials))',
        )
        ax.scatter(
            pareto['runtime'], pareto['accuracy'],
            color=color, marker=marker, s=60, zorder=5, edgecolors='white', linewidths=0.5,
        )
    
    ax.set_xlabel('Runtime (s)', fontsize=12)
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title(f'Pareto Front: Runtime vs Accuracy — {dataset}', fontsize=14)
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()